# MMA3001: CO2 spike, drift and bias detection

**Read the logic, then run from top to bottom.** This notebook contains no pretrained model. Training occurs when you execute the training cells.

Source CSV -> prepare and split chronologically -> create three test copies -> fit on original training rows -> calibrate on original validation rows -> detect faults in the copies -> evaluate using separate references and labels.

- **Spike:** one reading is suddenly too high or too low.
- **Drift:** the offset grows gradually over several readings.
- **Bias:** a constant offset is added throughout a window.

Cells 7-9 create faults separately. Cells 11-18 and 20-21 define detectors; cell 19 supplies CUSUM helpers. Cell 22 compares all methods, 23 compares prediction errors, 24 plots three local fault examples, and 25 downloads a private backup.

There are ten detectors and six distinct prediction curves. EWMA and CUSUM reuse the corresponding predictor; they are detection statistics, not new CO2 predictions.

An independent reference excludes the target sensor's CO2 history. Other channels from the same device may still be inaccurate. Both independent regression and independent tree use the same environmental inputs, including pressure when enabled.

Original records are not confirmed healthy. Simulated recall is not real-fault recall. This time period has already been inspected, so this is an exploratory study. Keep course measurements and private result archives out of public repositories.

Method references: [decision tree regression](https://scikit-learn.org/stable/modules/tree.html), [CUSUM](https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc323.htm).


**Code cell 1: Install and import dependencies**

Input: the Colab Python environment. Output: libraries used later. This cell does not read data or train a model.


In [ ]:
%pip -q install "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7" "scikit-learn>=1.5"

import hashlib
import json
import sys
import uuid
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from threadpoolctl import threadpool_limits
from IPython.display import display

print('Dependencies ready; no data have been read and no model has been trained.')

**Code cell 2: Shared experiment settings**

Choose magnitudes, event counts, durations and seeds. All methods follow these settings. Durations count readings, not minutes. After changes, rerun from this cell downward.


In [ ]:
# Leave blank to upload in Colab; enter your CSV path for local execution.
LOCAL_CSV = ""
TARGET_SENSOR = "6012002000869"

FAULT_SEED = 42           # Reproducible fault positions and signs
MODEL_SEED = 42           # Model initialization seed, separate from fault placement
SPIKE_PPM = 60.0
SPIKE_COUNT = 20
DRIFT_PPM = 60.0          # Absolute bias at the last injected drift reading
DRIFT_READINGS = 24
DRIFT_COUNT = 8
BIAS_PPM = 60.0           # Absolute constant-offset magnitude
BIAS_READINGS = 24
BIAS_COUNT = 8

MAX_GAP_MINUTES = 20
RECOVERY_READINGS = 30    # Leave unmodified readings after each event to assess delayed alarms.
THRESHOLD_QUANTILE = 0.99 # Common validation quantile; test alert rates need not match.
EWMA_ALPHA = 0.2
TREE_MAX_DEPTH = 5
TREE_MIN_SAMPLES_LEAF = 50
CUSUM_K_FRACTION = 0.1   # Allowance k = this fraction times the training residual SD.
CUSUM_H_GRID = np.arange(.5,40.5001,.25)  # Candidate h multiples, selected on validation only.
ADD_PRESSURE_TO_REFERENCE = True  # True requires complete pressure readings on the common cohort.

# Create a new directory for each run to preserve earlier results.
base_dir = Path("/content") if Path("/content").exists() else Path.cwd()
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:6]
RUN_DIR = base_dir / "private_co2_modular_runs" / run_id
print('Private output directory for this run:', RUN_DIR)
print('This cell only sets parameters; models are not trained yet.')

**Code cell 3: Upload the original extracted CSV**

Upload only env_6012002000869_all.csv. The notebook is self-contained; no extra Python files are required. Original means records without injected simulated faults.


In [ ]:
if LOCAL_CSV:
    INPUT_CSV = Path(LOCAL_CSV)
else:
    from google.colab import files
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError('Upload exactly one _all.csv containing CO2, temperature, humidity and pressure if requested.')
    INPUT_CSV = Path(csv_names[0])
    del uploaded  # Read the saved file later and release the uploaded bytes.

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

SOURCE_HASH = file_sha256(INPUT_CSV)
print('Input file:', INPUT_CSV.name)

**Code cell 4: Prepare one row per timestamp**

Read the long-format CSV. Remove identical duplicates; reject conflicting values. Keep the common complete CO2/temperature/humidity cohort without interpolation. Required pressure must be complete on it.


In [ ]:
raw = pd.read_csv(INPUT_CSV, dtype={"sensor_id": str, "variable": str})
required = {"sensor_id", "timestamp_utc", "variable", "value"}
if not required.issubset(raw.columns):
    raise ValueError('CSV must contain sensor_id, timestamp_utc, variable and value.')

needed = ["Carbon dioxide", "Temperature", "Humidity"]
if ADD_PRESSURE_TO_REFERENCE:
    needed.append("Pressure")
selected = raw[(raw.sensor_id == TARGET_SENSOR) & raw.variable.isin(needed)].copy()
if set(needed)-set(selected.variable):
    raise ValueError('Required variables are missing; set ADD_PRESSURE_TO_REFERENCE=False if only the three basic channels are available.')
selected["timestamp_utc"] = pd.to_datetime(selected.timestamp_utc, utc=True, errors="coerce")
selected["value"] = pd.to_numeric(selected.value, errors="coerce")
if selected.timestamp_utc.isna().any() or not np.isfinite(selected.value.to_numpy()).all():
    raise ValueError('Selected records contain invalid timestamps or values; inspect the source.')
keys = ["timestamp_utc", "variable"]
if (selected.groupby(keys).value.nunique()>1).any():
    raise ValueError('Conflicting values exist for the same timestamp and variable; investigate first.')
duplicates_removed = int(selected.duplicated(keys).sum())
selected = selected.drop_duplicates(keys)
pivot = selected.pivot(index="timestamp_utc", columns="variable", values="value").sort_index()
original = pivot.dropna(subset=["Carbon dioxide", "Temperature", "Humidity"]).copy()
if ADD_PRESSURE_TO_REFERENCE and original.Pressure.isna().any():
    raise ValueError('Pressure is missing on the common cohort. Set ADD_PRESSURE_TO_REFERENCE=False to retain that cohort.')
if len(original)<500:
    raise ValueError('At least 500 complete timestamps are required.')
audit = {"sensor_id":TARGET_SENSOR,"complete_timestamps":len(original),
         "identical_duplicates_removed":duplicates_removed,
         "source_filename":INPUT_CSV.name,"source_sha256":SOURCE_HASH}
print('Complete distinct timestamps:', len(original))
display(original.head(3))
print('This step only prepares the data; no training has occurred.')

**Code cell 5: Build features and split chronologically**

Output: training, validation and test indices. Training learns parameters; validation sets thresholds; test copies receive faults. This cell only prepares and splits data, without training.


In [ ]:
def make_features(observed):
    # Build inputs from observations; injected drift also contaminates these lags.
    co2 = observed["Carbon dioxide"]
    hour = observed.index.hour+observed.index.minute/60+observed.index.second/3600
    x = pd.DataFrame(index=observed.index)
    x["co2_lag1"] = co2.shift(1)  # Previous reading
    x["co2_lag6"] = co2.shift(6)  # Sixth previous reading, not necessarily one hour earlier
    x["temperature"] = observed["Temperature"]
    x["humidity"] = observed["Humidity"]
    x["hour_sin"] = np.sin(2*np.pi*hour/24)
    x["hour_cos"] = np.cos(2*np.pi*hour/24)
    x["weekend"] = (observed.index.dayofweek>=5).astype(float)  # Keep the existing UTC encoding.
    x["minutes_since_previous"] = observed.index.to_series().diff().dt.total_seconds()/60
    if "Pressure" in observed:
        x["pressure"] = observed["Pressure"]
    return x

HISTORY_FEATURES = ["co2_lag1","co2_lag6","temperature","humidity",
                    "hour_sin","hour_cos","weekend","minutes_since_previous"]
REFERENCE_FEATURES = [c for c in HISTORY_FEATURES if not c.startswith("co2_lag")]
if ADD_PRESSURE_TO_REFERENCE:
    REFERENCE_FEATURES.append("pressure")

original_features = make_features(original)
gaps = original.index.to_series().diff().dt.total_seconds()/60
# Require complete basic channels and six past readings without a large gap.
eligible = original_features[HISTORY_FEATURES].notna().all(axis=1) & (gaps.rolling(6).max()<=MAX_GAP_MINUTES)
ELIGIBLE_IDX = original.index[eligible]
if len(ELIGIBLE_IDX)<500:
    raise ValueError('Too few rows satisfy the history and continuity requirements.')
n_train, n_train_val = int(.6*len(ELIGIBLE_IDX)), int(.8*len(ELIGIBLE_IDX))
TRAIN_IDX = ELIGIBLE_IDX[:n_train]
VALIDATION_IDX = ELIGIBLE_IDX[n_train:n_train_val]
TEST_IDX = ELIGIBLE_IDX[n_train_val:]
split = pd.Series("excluded",index=original.index,name="split")
split.loc[TRAIN_IDX],split.loc[VALIDATION_IDX],split.loc[TEST_IDX] = "train","validation","test"
MONITORED_IDX = VALIDATION_IDX.append(TEST_IDX)
train_y = original.loc[TRAIN_IDX,"Carbon dioxide"]
# Calibrate on this prefix, which contains no test timestamps.
calibration_observations = original.loc[:VALIDATION_IDX[-1]].copy()
print(f"Train {len(TRAIN_IDX):,}; validation {len(VALIDATION_IDX):,}; test {len(TEST_IDX):,}; excluded {(split=='excluded').sum():,}")
print('Independent reference inputs:', REFERENCE_FEATURES)
print('These are split sizes; models are not trained yet.')

**Code cell 6: Shared fault-window planner**

Input: eligible test timestamps, event count and duration. Output: separated continuous event windows. This cell selects locations; cells 7-9 add the actual offsets.


In [ ]:
def plan_windows(duration, count, seed):
    positions = np.flatnonzero(split.to_numpy()=="test")
    candidates = []
    for start in range(6,len(positions)-duration-RECOVERY_READINGS+1):
        context = positions[start-6:start+duration+RECOVERY_READINGS]
        if not np.all(np.diff(context)==1):
            continue
        times = original.index[context]
        if (times.to_series().diff().dt.total_seconds().dropna()/60<=MAX_GAP_MINUTES).all():
            candidates.append(start)
    slots,next_allowed = [],0
    for start in candidates:
        if start>=next_allowed:
            slots.append(start)
            next_allowed=start+duration+RECOVERY_READINGS
    if count<1 or duration<1 or len(slots)<count:
        raise ValueError(f"Only {len(slots)} usable windows found; reduce event count or duration.")
    rng=np.random.default_rng(seed)
    chosen=sorted(rng.choice(slots,size=count,replace=False).tolist())
    events=[]
    for event_id,start in enumerate(chosen,1):
        active=original.index[positions[start:start+duration]]
        recovery=original.index[positions[start+duration:start+duration+RECOVERY_READINGS]]
        events.append({"event_id":event_id,"sign":int(rng.choice([-1,1])),
                       "active":active,"recovery":recovery})
    return events

def save_scenario(name, observed, offsets, events):
    # Store the evaluation reference separately; detectors never receive it.
    reference=pd.DataFrame({"original_co2_ppm":original["Carbon dioxide"],
                            "offset_ppm":offsets,"event_id":0},index=original.index)
    for event in events:
        reference.loc[event["active"],"event_id"]=event["event_id"]
    assert (offsets.loc[split!="test"]==0).all()
    pd.testing.assert_frame_equal(observed.loc[split!="test"],original.loc[split!="test"])
    pd.testing.assert_frame_equal(observed.drop(columns="Carbon dioxide"),original.drop(columns="Carbon dioxide"))
    np.testing.assert_allclose(observed["Carbon dioxide"]-original["Carbon dioxide"],offsets,atol=1e-10,rtol=0)
    folder=RUN_DIR/"fault_data"/name
    folder.mkdir(parents=True,exist_ok=False)
    observations=observed.reset_index().copy()
    observations["split"]=split.to_numpy()
    observations.to_csv(folder/"observations.csv",index=False)
    reference.to_csv(folder/"reference_labels.csv")
    events_saved=[{"event_id":e["event_id"],"sign":e["sign"],
                   "active_timestamps_utc":[t.isoformat() for t in e["active"]],
                   "recovery_timestamps_utc":[t.isoformat() for t in e["recovery"]]} for e in events]
    manifest={"kind":name,"events":events_saved,"source_sha256":SOURCE_HASH,
              "fault_seed":FAULT_SEED,"recovery_readings":RECOVERY_READINGS,
              "note":'Only CO2 bias is simulated; the original health status is unknown.'}
    (folder/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print(name,":",len(events),'events; only test copies are modified, and evaluation references are stored separately.')
    return {"name":name,"observations":observed,"reference":reference,"events":events,"folder":folder}

scenarios={}
print('Window planner ready; no faults have been created and no model has been trained yet.')

**Code cell 7: Create spike faults separately**

Add a signed offset to one reading per event. Output: a corrupted test copy and separate evaluation labels. No model is trained.


In [ ]:
spike_events=plan_windows(duration=1,count=SPIKE_COUNT,seed=FAULT_SEED)
spike_offsets=pd.Series(0.0,index=original.index)
for event in spike_events:
    # One reading per spike; random signs with equal magnitudes.
    spike_offsets.loc[event["active"]]=event["sign"]*SPIKE_PPM
spike_observed=original.copy()
spike_observed["Carbon dioxide"]+=spike_offsets
scenarios["spike"]=save_scenario("spike",spike_observed,spike_offsets,spike_events)

**Code cell 8: Create drift faults separately**

Increase the signed offset linearly to +/-DRIFT_PPM. This finite ramp returns to the original record afterward; it does not cover every form of permanent sensor drift.


In [ ]:
drift_events=plan_windows(duration=DRIFT_READINGS,count=DRIFT_COUNT,seed=FAULT_SEED)
drift_offsets=pd.Series(0.0,index=original.index)
for event in drift_events:
    # With 24 readings and 60 ppm, offsets are +/-2.5, +/-5, ..., +/-60 ppm.
    ramp=np.arange(1,DRIFT_READINGS+1)/DRIFT_READINGS
    drift_offsets.loc[event["active"]]=event["sign"]*DRIFT_PPM*ramp
drift_observed=original.copy()
drift_observed["Carbon dioxide"]+=drift_offsets
scenarios["drift"]=save_scenario("drift",drift_observed,drift_offsets,drift_events)

**Code cell 9: Create bias faults separately**

Add the same signed offset throughout each active window. Unlike drift, the offset does not grow. Matching seeds, counts and durations give bias and drift the same event positions.


In [ ]:
bias_events=plan_windows(duration=BIAS_READINGS,count=BIAS_COUNT,seed=FAULT_SEED)
bias_offsets=pd.Series(0.0,index=original.index)
for event in bias_events:
    # Every active reading has the same +60 or -60 offset; entry and exit are steps.
    bias_offsets.loc[event["active"]]=event["sign"]*BIAS_PPM
bias_observed=original.copy()
bias_observed["Carbon dioxide"]+=bias_offsets
scenarios["bias"]=save_scenario("bias",bias_observed,bias_offsets,bias_events)

**Code cell 10: Shared OLS and threshold helpers**

The helpers fit OLS using only original training X/y and register methods using original validation scores. Actual method construction follows in later cells.


In [ ]:
def fit_ols(train_x, y):
    # train_x and y come from original training rows; scaling uses only these rows.
    mean=train_x.mean().to_numpy()
    scale=train_x.std(ddof=0).to_numpy().copy()
    scale[scale==0]=1
    design=np.column_stack([np.ones(len(train_x)),(train_x.to_numpy()-mean)/scale])
    coefficients,_,rank,_=np.linalg.lstsq(design,y.to_numpy(),rcond=None)
    if rank<design.shape[1]:
        raise ValueError('Training features are constant or linearly dependent; inspect or remove the affected feature.')
    return {"columns":list(train_x.columns),"mean":mean,"scale":scale,"coefficients":coefficients}

def predict_ols(model,observed):
    # Receive observations, including corrupted copies at test time, never references.
    available=ELIGIBLE_IDX.intersection(observed.index)
    x=make_features(observed).loc[available,model["columns"]]
    design=np.column_stack([np.ones(len(x)),(x.to_numpy()-model["mean"])/model["scale"]])
    return pd.Series(design@model["coefficients"],index=available)

def residual_score(observed,predictor):
    prediction=predictor(observed)
    return observed.loc[prediction.index,"Carbon dioxide"]-prediction

methods={}
model_metadata={}
def register_method(key,label,predictor,score_function,score_label):
    # Calibrate only on original validation; the prefix has no test timestamps.
    validation_score=score_function(calibration_observations).loc[VALIDATION_IDX]
    if not np.isfinite(validation_score.to_numpy()).all():
        raise ValueError('Validation scores contain missing or nonfinite values.')
    threshold=float(validation_score.abs().quantile(THRESHOLD_QUANTILE))
    methods[key]={"label":label,"predictor":predictor,"score_function":score_function,
                  "threshold_ppm":threshold,"score_label":score_label,
                  "threshold_calibration":"validation absolute-score quantile",
                  "validation_alert_rate":float((validation_score.abs()>threshold).mean())}
    print(label,': validation threshold',round(threshold,3),'ppm; fault labels are not used for calibration.')

print('Helpers ready; method construction starts in the next cell.')

**Code cell 11: Method 1: adjacent change rule**

No regression training is needed. Alarm when the change from the previous reading is large. Its prediction/reference is the previous observed reading (persistence).


In [ ]:
def predict_persistence(observed):
    available=ELIGIBLE_IDX.intersection(observed.index)
    return observed["Carbon dioxide"].shift(1).loc[available]

def score_change_rule(observed):
    return residual_score(observed,predict_persistence)

register_method("change_rule",'Adjacent change rule',predict_persistence,score_change_rule,'Adjacent change')

**Code cell 12: Method 2: history CO2 linear regression**

Training actually occurs here, using original training history, environment and time features. Test prediction rebuilds lagged inputs from the corrupted observations.


In [ ]:
# Learn coefficients from original training data, not fault copies.
history_ols=fit_ols(original_features.loc[TRAIN_IDX,HISTORY_FEATURES],train_y)

def predict_history_regression(observed):
    return predict_ols(history_ols,observed)

def score_history_regression(observed):
    return residual_score(observed,predict_history_regression)

model_metadata["history_regression"]={"features":HISTORY_FEATURES,
    "standardized_coefficients":history_ols["coefficients"].tolist(),"fit_on":"original training only"}
register_method("history_regression",'History CO2 regression',predict_history_regression,score_history_regression,'Regression residual')

**Code cell 13: Method 3: small history CO2 neural network**

Use the same eight inputs as history regression. Fit input and target scaling on training only. This is a predictor, not a classifier trained on fault labels; drift detection is not guaranteed.


In [ ]:
# Eight inputs, hidden layers of 16 and 8 units, one CO2 prediction.
nn_train_x=original_features.loc[TRAIN_IDX,HISTORY_FEATURES].to_numpy()
nn_x_mean=nn_train_x.mean(axis=0)
nn_x_scale=nn_train_x.std(axis=0)
nn_x_scale[nn_x_scale==0]=1
nn_y_mean,nn_y_scale=float(train_y.mean()),float(train_y.std(ddof=0))
if nn_y_scale<=0:
    raise ValueError('Training CO2 is constant; the neural network requires a varying target.')
neural_network=MLPRegressor(hidden_layer_sizes=(16,8),activation="relu",solver="adam",
    alpha=.01,batch_size=128,learning_rate_init=.001,max_iter=400,shuffle=False,
    random_state=MODEL_SEED,tol=1e-4,early_stopping=False,n_iter_no_change=20)
# Explicit training call using original training X and original training y.
with warnings.catch_warnings(record=True) as fit_warnings,threadpool_limits(limits=1):
    warnings.simplefilter("always")
    neural_network.fit((nn_train_x-nn_x_mean)/nn_x_scale,(train_y.to_numpy()-nn_y_mean)/nn_y_scale)

def predict_history_neural(observed):
    available=ELIGIBLE_IDX.intersection(observed.index)
    x=make_features(observed).loc[available,HISTORY_FEATURES].to_numpy()
    with threadpool_limits(limits=1):
        scaled_prediction=neural_network.predict((x-nn_x_mean)/nn_x_scale)
    return pd.Series(nn_y_mean+nn_y_scale*scaled_prediction,index=available)

def score_history_neural(observed):
    return residual_score(observed,predict_history_neural)

nn_arrays=neural_network.coefs_+neural_network.intercepts_
model_metadata["history_neural"]={"features":HISTORY_FEATURES,"epochs":int(neural_network.n_iter_),
    "parameters":sum(a.size for a in nn_arrays),"settings":neural_network.get_params(),
    "weights_sha256":hashlib.sha256(b"".join(a.tobytes() for a in nn_arrays)).hexdigest(),
    "warnings":[str(w.message) for w in fit_warnings],"fit_on":"original training only"}
print('NN training completed; epochs:',neural_network.n_iter_)
for item in fit_warnings:
    print('Training warning:',str(item.message))
register_method("history_neural",'History CO2 neural network',predict_history_neural,score_history_neural,'NN residual')

**Code cell 14: Method 4: independent environmental regression**

Inputs are temperature, humidity, time, sampling interval and optional pressure. The target is original training CO2. Excluding target CO2 history prevents injected bias from changing its prediction.


In [ ]:
independent_ols=fit_ols(original_features.loc[TRAIN_IDX,REFERENCE_FEATURES],train_y)

def predict_independent_regression(observed):
    return predict_ols(independent_ols,observed)

def score_independent_regression(observed):
    return residual_score(observed,predict_independent_regression)

model_metadata["independent_regression"]={"features":REFERENCE_FEATURES,
    "standardized_coefficients":independent_ols["coefficients"].tolist(),"fit_on":"original training only"}
register_method("independent_regression",'Independent environmental regression',predict_independent_regression,
                score_independent_regression,'Independent reference residual')

**Code cell 15: Method 5: history regression residual EWMA**

Reuse the fitted history regression without retraining. Monitor a signed residual average over time. EWMA is the alarm statistic; the CO2 prediction still comes from regression.


In [ ]:
def ewma_series(residual,alpha):
    values,state,previous=[],0.0,None
    for timestamp,value in residual.items():
        if previous is None or (timestamp-previous).total_seconds()/60>MAX_GAP_MINUTES:
            state=0.0
        state=alpha*float(value)+(1-alpha)*state
        values.append(state)
        previous=timestamp
    return pd.Series(values,index=residual.index)

history_residual_center=float(score_history_regression(original.loc[:TRAIN_IDX[-1]]).loc[TRAIN_IDX].mean())
def score_history_ewma(observed):
    residual=score_history_regression(observed)
    monitored=MONITORED_IDX.intersection(residual.index)
    # z_t = 0.2*(e_t-training mean) + 0.8*z_(t-1)
    # Carry validation state into test; never reset from event labels.
    return ewma_series(residual.loc[monitored]-history_residual_center,EWMA_ALPHA)

register_method("history_regression_ewma",'History regression residual EWMA',predict_history_regression,
                score_history_ewma,'EWMA statistic')

**Code cell 16: Method 6: independent reference residual EWMA**

Reuse the independent regression. The only reference change from method 5 is the predictor, allowing comparison of history-dependent and independent residual monitoring.


In [ ]:
independent_residual_center=float(score_independent_regression(original.loc[:TRAIN_IDX[-1]]).loc[TRAIN_IDX].mean())
def score_independent_ewma(observed):
    residual=score_independent_regression(observed)
    monitored=MONITORED_IDX.intersection(residual.index)
    return ewma_series(residual.loc[monitored]-independent_residual_center,EWMA_ALPHA)

register_method("independent_regression_ewma",'Independent reference residual EWMA',predict_independent_regression,
                score_independent_ewma,'EWMA statistic')

**Code cell 17: Method 7: history CO2 decision tree regression**

Use the same eight inputs as history regression/NN. Fit branching rules on original training rows. Depth and minimum leaf size are fixed before fault evaluation; alarm on prediction residuals.


In [ ]:
history_tree=DecisionTreeRegressor(max_depth=TREE_MAX_DEPTH,
    min_samples_leaf=TREE_MIN_SAMPLES_LEAF,random_state=MODEL_SEED)
# Explicit training call; fault copies and fault labels are excluded.
history_tree.fit(original_features.loc[TRAIN_IDX,HISTORY_FEATURES],train_y)

def predict_history_tree(observed):
    available=ELIGIBLE_IDX.intersection(observed.index)
    x=make_features(observed).loc[available,HISTORY_FEATURES]
    return pd.Series(history_tree.predict(x),index=available)

def score_history_tree(observed):
    return residual_score(observed,predict_history_tree)

model_metadata["history_tree"]={"features":HISTORY_FEATURES,"settings":history_tree.get_params(),
    "depth":int(history_tree.get_depth()),"leaves":int(history_tree.get_n_leaves()),
    "fit_on":"original training only"}
register_method("history_tree",'History CO2 decision tree',predict_history_tree,score_history_tree,'Tree regression residual')

**Code cell 18: Method 8: independent environmental decision tree**

Use exactly the same environmental features as independent regression, including optional pressure. No target CO2 history enters prediction, and the sample cohort is unchanged.


In [ ]:
independent_tree=DecisionTreeRegressor(max_depth=TREE_MAX_DEPTH,
    min_samples_leaf=TREE_MIN_SAMPLES_LEAF,random_state=MODEL_SEED)
independent_tree.fit(original_features.loc[TRAIN_IDX,REFERENCE_FEATURES],train_y)

def predict_independent_tree(observed):
    available=ELIGIBLE_IDX.intersection(observed.index)
    x=make_features(observed).loc[available,REFERENCE_FEATURES]
    return pd.Series(independent_tree.predict(x),index=available)

def score_independent_tree(observed):
    return residual_score(observed,predict_independent_tree)

model_metadata["independent_tree"]={"features":REFERENCE_FEATURES,"settings":independent_tree.get_params(),
    "depth":int(independent_tree.get_depth()),"leaves":int(independent_tree.get_n_leaves()),
    "fit_on":"original training only"}
register_method("independent_tree",'Independent environmental decision tree',predict_independent_tree,
                score_independent_tree,'Independent tree reference residual')

**Code cell 19: CUSUM helpers: accumulate small residual shifts**

CUSUM does not produce another CO2 prediction. Accumulate positive and negative evidence separately. Reset after an alarm or a large gap. Derive k from training residuals and select h on original validation only.


In [ ]:
def cusum_series(residual,k,h):
    if not 0<=k or not h>0 or not np.isfinite([k,h]).all():
        raise ValueError('CUSUM requires finite k>=0 and h>0.')
    if not np.isfinite(residual.to_numpy()).all():
        raise ValueError('CUSUM residuals must be finite.')
    scores=[]
    up=down=0.0
    previous=None
    for timestamp,value in residual.items():
        if previous is None or (timestamp-previous).total_seconds()/60>MAX_GAP_MINUTES:
            up=down=0.0
        # Accumulate each direction separately; subtract allowance k at every step.
        up=max(0.0,up+float(value)-k)
        down=max(0.0,down-float(value)-k)
        score=max(up,down)
        scores.append(score)  # Determine the current alarm from the score before reset.
        if score>h:
            up=down=0.0      # Reset on the alarm, without reading fault-end labels.
        previous=timestamp
    return pd.Series(scores,index=residual.index)

def build_cusum_method(key,label,predictor,base_score_function):
    training_residual=base_score_function(original.loc[:TRAIN_IDX[-1]]).loc[TRAIN_IDX]
    center=float(training_residual.mean())
    sigma=float(training_residual.std(ddof=0))
    if not sigma>0 or not 0<CUSUM_K_FRACTION:
        raise ValueError('CUSUM requires varying training residuals and a positive k fraction.')
    k=CUSUM_K_FRACTION*sigma
    validation_residual=base_score_function(calibration_observations).loc[VALIDATION_IDX]-center
    # Match the validation alert budget: at most ceil(1%*2458)=25 by default.
    budget=int(np.ceil((1-THRESHOLD_QUANTILE)*len(VALIDATION_IDX)))
    chosen=None
    trials=[]
    for multiplier in CUSUM_H_GRID:
        h=float(multiplier*sigma)
        score=cusum_series(validation_residual,k,h)
        alerts=int((score>h).sum())
        trials.append({"h":h,"validation_alerts":alerts})
        if alerts<=budget:
            chosen=(h,alerts)
            break  # Select the smallest candidate meeting the validation budget, not the best test result.
    if chosen is None:
        raise ValueError('No CUSUM threshold candidate meets the validation alert budget.')
    h,validation_alerts=chosen

    def score_function(observed):
        residual=base_score_function(observed)
        monitored=MONITORED_IDX.intersection(residual.index)
        # Carry validation state into test; never reset from event positions.
        return cusum_series(residual.loc[monitored]-center,k,h)

    methods[key]={"label":label,"predictor":predictor,"score_function":score_function,
        "threshold_ppm":h,"score_label":"CUSUM statistic",
        "threshold_calibration":"validation alert-budget search",
        "validation_alert_rate":validation_alerts/len(VALIDATION_IDX)}
    model_metadata[key]={"training_residual_center_ppm":center,"training_residual_sd_ppm":sigma,
        "k_fraction":CUSUM_K_FRACTION,"k":k,"h":h,"validation_alert_budget":budget,
        "validation_alerts":validation_alerts,"threshold_trials":trials,
        "reset_after_alarm":True,"fit_on":"training residuals and original validation only"}
    print(label,": k =",round(k,3),"；h =",round(h,3),'; validation alerts =',validation_alerts)

print('CUSUM helpers ready; existing predictions are reused without retraining predictors.')

**Code cell 20: Method 9: history regression residual CUSUM**

Reuse the regression from cell 12. Center residuals on the training mean and accumulate directional shifts. The cumulative threshold h is not comparable to a single-reading residual threshold.


In [ ]:
build_cusum_method("history_regression_cusum",'History regression residual CUSUM',
                   predict_history_regression,score_history_regression)

**Code cell 21: Method 10: history NN residual CUSUM**

Reuse the NN from cell 13. Greater sensitivity to persistent shifts may increase background alarms. Fix the k fraction and calibrate h without test fault labels.


In [ ]:
build_cusum_method("history_neural_cusum",'History NN residual CUSUM',
                   predict_history_neural,score_history_neural)
threshold_table=pd.DataFrame([{"method":key,"name":m["label"],"threshold_ppm":m["threshold_ppm"],
    "threshold_calibration":m["threshold_calibration"],"validation_alert_rate":m["validation_alert_rate"]}
    for key,m in methods.items()])
display(threshold_table.round(4))
print('Point and EWMA thresholds use a validation quantile; CUSUM searches h under a comparable validation alert budget.')

**Code cell 22: Integrate ten detectors across three fault types**

First infer alarms using corrupted observations alone. Then evaluate against references and labels. Compare event recall, paired additional alarms, background alerts, recovery alerts and delays.


In [ ]:
def infer_scenario(observations):
    # This function receives observations, not references, event positions or original test CO2.
    inferred={}
    for key,method in methods.items():
        prediction=method["predictor"](observations).loc[TEST_IDX]
        score=method["score_function"](observations).loc[TEST_IDX]
        if not np.isfinite(score.to_numpy()).all():
            raise ValueError('Test scores contain nonfinite values.')
        inferred[key]=pd.DataFrame({"prediction_ppm":prediction,"score_ppm":score,
                                   "alarm":score.abs()>method["threshold_ppm"]},index=TEST_IDX)
    return inferred

# Infer all fault-copy alarms first; use original values and labels only for evaluation.
inferred_by_scenario={name:infer_scenario(scenario["observations"]) for name,scenario in scenarios.items()}
unmodified_inference=infer_scenario(original)  # Evaluation baseline, not an input to fault-copy prediction
summary_rows,event_rows=[],[]
for name,scenario in scenarios.items():
    reference=scenario["reference"].loc[TEST_IDX]  # Only now used for evaluation
    injected=reference.event_id>0
    recovery=pd.Series(False,index=TEST_IDX)
    for event in scenario["events"]:
        recovery.loc[event["recovery"]]=True
    outside=~injected & ~recovery
    result_folder=RUN_DIR/"results"/name
    result_folder.mkdir(parents=True,exist_ok=False)
    for key,points in inferred_by_scenario[name].items():
        alarm=points.alarm
        original_alarm=unmodified_inference[key].alarm
        detected,additional,delays=0,0,[]
        for event in scenario["events"]:
            active=event["active"]
            hit_times=active[alarm.loc[active].to_numpy()]
            detected_flag=len(hit_times)>0
            additional_flag=bool((alarm.loc[active]&~original_alarm.loc[active]).any())
            delay=(hit_times[0]-active[0]).total_seconds()/60 if detected_flag else None
            detected+=int(detected_flag)
            additional+=int(additional_flag)
            if delay is not None:
                delays.append(delay)
            event_rows.append({"scenario":name,"method":key,"event_id":event["event_id"],
                               "detected":detected_flag,"additional_alarm":additional_flag,
                               "first_alert_delay_minutes":delay})
        total=len(scenario["events"])
        summary_rows.append({"scenario":name,"method":key,"name":methods[key]["label"],
            "events":total,"detected":detected,"event_recall":detected/total,
            "paired_additional_events":additional,"paired_additional_event_fraction":additional/total,
            "reading_recall":float(alarm.loc[injected].mean()),
            "unmodified_test_alert_rate":float(original_alarm.mean()),
            "outside_fault_and_recovery_alert_rate":float(alarm.loc[outside].mean()) if outside.any() else None,
            "recovery_alert_rate":float(alarm.loc[recovery].mean()),
            "median_first_alert_delay_minutes":float(np.median(delays)) if delays else None})
        exported=points.copy()
        exported.insert(0,"observed_co2_ppm",scenario["observations"].loc[TEST_IDX,"Carbon dioxide"])
        exported["original_co2_ppm"]=reference.original_co2_ppm
        exported["event_id"]=reference.event_id
        exported["threshold_ppm"]=methods[key]["threshold_ppm"]
        exported.to_csv(result_folder/f"{key}_points.csv")

comparison=pd.DataFrame(summary_rows)
event_comparison=pd.DataFrame(event_rows)
comparison.to_csv(RUN_DIR/"comparison_all_methods.csv",index=False)
event_comparison.to_csv(RUN_DIR/"event_results.csv",index=False)
# Show the main columns first; the CSV stores all metrics.
display(comparison[["scenario","name","detected","events","paired_additional_events",
                    "unmodified_test_alert_rate"]].round(4))
print('detected: events with an alarm during injection; paired_additional_events: events with an added alarm relative to the unmodified copy at the same timestamps.')
print('Unmodified alert rates describe background alerts, not verified false-positive rates; original health is unknown.')

**Code cell 23: Separate prediction error from detection performance**

Compare predictions against unmodified records and pre-injection references during faults. Evaluate six distinct predictors; EWMA/CUSUM reuse predictions and are not counted as extra prediction models.


In [ ]:
PREDICTION_METHODS=["change_rule","history_regression","history_neural","independent_regression",
                    "history_tree","independent_tree"]
prediction_rows=[]
for key in PREDICTION_METHODS:
    original_prediction=methods[key]["predictor"](original)
    for part,index in (("train",TRAIN_IDX),("validation",VALIDATION_IDX),("test_unmodified",TEST_IDX)):
        error=original.loc[index,"Carbon dioxide"]-original_prediction.loc[index]
        prediction_rows.append({"method":key,"name":methods[key]["label"],"part":part,
                                "mae_ppm":float(error.abs().mean()),"rmse_ppm":float(np.sqrt(np.mean(error**2)))})
prediction_quality=pd.DataFrame(prediction_rows)
prediction_quality.to_csv(RUN_DIR/"prediction_quality.csv",index=False)
display(prediction_quality.pivot(index="name",columns="part",values="mae_ppm").round(2))
print('The table reports prediction MAE; accurate prediction does not guarantee drift detection.')

injected_prediction_rows=[]
for name,scenario in scenarios.items():
    active=TEST_IDX[scenario["reference"].loc[TEST_IDX,"event_id"]>0]
    baseline=scenario["reference"].loc[active,"original_co2_ppm"]
    for key in PREDICTION_METHODS:
        prediction=inferred_by_scenario[name][key].loc[active,"prediction_ppm"]
        error=prediction-baseline
        injected_prediction_rows.append({"scenario":name,"method":key,
            "prediction_vs_original_injected_mae_ppm":float(error.abs().mean()),
            "prediction_vs_original_injected_rmse_ppm":float(np.sqrt(np.mean(error**2)))})
injected_prediction_quality=pd.DataFrame(injected_prediction_rows)
injected_prediction_quality.to_csv(RUN_DIR/"prediction_vs_original_injected.csv",index=False)
print('Prediction errors against the pre-injection record are saved in prediction_vs_original_injected.csv.')

**Code cell 24: Three fault panels, six predictions, one shared legend**

Show one event of each fault type with six readings of context on each side. Colors and CO2 limits are shared; relative time ranges are separate. Default red crosses are NN-CUSUM alarms. Rerun this cell to change views without retraining.


In [ ]:
PLOT_SCENARIOS=["spike","drift","bias"]  # Use ["drift"], for example, to show just one panel.
PLOT_EVENT_IDS={"spike":1,"drift":1,"bias":1}  # Select one event for each fault type.
ALARM_METHOD="history_neural_cusum"  # Controls red crosses only; all six distinct predictions remain visible.
POINTS_BEFORE=6
POINTS_AFTER=6
SHOW_ALARM_TABLE=True

if not PLOT_SCENARIOS or len(set(PLOT_SCENARIOS))!=len(PLOT_SCENARIOS):
    raise ValueError('Select at least one fault type, with no duplicates.')
if any(name not in scenarios for name in PLOT_SCENARIOS) or ALARM_METHOD not in methods:
    raise ValueError('Check the fault types or ALARM_METHOD name.')
if POINTS_BEFORE<0 or POINTS_AFTER<0:
    raise ValueError('Context point counts must be nonnegative.')

prediction_styles={
    "change_rule":("tab:brown",":","Previous reading (change rule)"),
    "history_regression":("tab:blue","--","History regression / EWMA & CUSUM reference"),
    "history_neural":("tab:purple","-.","History neural network / CUSUM reference"),
    "independent_regression":("tab:green","--","Independent regression / EWMA reference"),
    "history_tree":("tab:cyan",(0,(4,1)),"History decision tree"),
    "independent_tree":("tab:olive",(0,(1,2)),"Independent decision tree"),
}
# Prepare local events from cell 22 results without training models.
panels=[]
for name in PLOT_SCENARIOS:
    scenario=scenarios[name]
    event_id=PLOT_EVENT_IDS.get(name,1)
    event=next((e for e in scenario["events"] if e["event_id"]==event_id),None)
    if event is None:
        raise ValueError(f"{name} has no event {event_id}; check PLOT_EVENT_IDS.")
    first,last=TEST_IDX.get_loc(event["active"][0]),TEST_IDX.get_loc(event["active"][-1])
    local=TEST_IDX[max(0,first-POINTS_BEFORE):min(len(TEST_IDX),last+POINTS_AFTER+1)]
    start=event["active"][0]
    minutes=(local-start).total_seconds()/60  # Zero is the first injected timestamp for this event.
    panels.append({"name":name,"event_id":event_id,"event":event,"scenario":scenario,
        "local":local,"minutes":minutes,
        "observed":scenario["observations"].loc[local,"Carbon dioxide"],
        "reference":scenario["reference"].loc[local,"original_co2_ppm"],
        "results":inferred_by_scenario[name]})

height=3.0*len(panels)+1.7
fig,axes_grid=plt.subplots(len(panels),1,figsize=(11,height),sharey=True,squeeze=False)
axes=axes_grid[:,0]
plot_prediction_rows,plot_alarm_rows,y_values=[],[],[]
for ax,panel in zip(axes,panels):
    name,event_id,event=panel["name"],panel["event_id"],panel["event"]
    local,minutes=panel["local"],panel["minutes"]
    observed,reference=panel["observed"],panel["reference"]
    results=panel["results"]
    # Gray observed = original plus simulated offset; black is an evaluation reference.
    ax.plot(minutes,reference,color="black",lw=2,label="Original record (pre-injection)")
    ax.plot(minutes,observed,color="0.65",lw=1.6,label="Observed (simulated fault)")
    y_values.extend([reference.to_numpy(),observed.to_numpy()])
    for key,(color,style,label) in prediction_styles.items():
        prediction=results[key].loc[local,"prediction_ppm"]
        ax.plot(minutes,prediction,color=color,ls=style,lw=1.6,label=label)
        y_values.append(prediction.to_numpy())
        error=prediction-reference
        active_error=results[key].loc[event["active"],"prediction_ppm"]-panel["scenario"]["reference"].loc[event["active"],"original_co2_ppm"]
        plot_prediction_rows.append({"scenario":name,"event_id":event_id,"method":key,
            "name":methods[key]["label"],"local_readings":len(local),
            "local_mae_vs_original_ppm":float(error.abs().mean()),
            "injected_mae_vs_original_ppm":float(active_error.abs().mean())})

    alarm_points=results[ALARM_METHOD].loc[local]
    hits=local[alarm_points.alarm.to_numpy()]
    hit_minutes=(hits-event["active"][0]).total_seconds()/60
    ax.scatter(hit_minutes,observed.loc[hits],color="red",marker="x",s=55,zorder=5,
               label=f"Alarm: {ALARM_METHOD}")
    for number,(timestamp,x_value) in enumerate(zip(hits,hit_minutes),1):
        ax.annotate(f"A{number}",(x_value,observed.loc[timestamp]),xytext=(6,6),
                    textcoords="offset points",color="red",fontsize=8)
        plot_alarm_rows.append({"scenario":name,"event_id":event_id,"alarm_id":f"A{number}",
            "minutes_from_start":float(x_value),"observed_ppm":float(observed.loc[timestamp]),
            "prediction_ppm":float(alarm_points.loc[timestamp,"prediction_ppm"]),
            "score_ppm":float(alarm_points.loc[timestamp,"score_ppm"]),
            "threshold_ppm":methods[ALARM_METHOD]["threshold_ppm"],
            "inside_injected_event":bool(panel["scenario"]["reference"].loc[timestamp,"event_id"]==event_id)})
    end_minutes=(event["active"][-1]-event["active"][0]).total_seconds()/60
    if len(event["active"])==1:
        ax.axvline(0,color="orange",lw=1.5,label="Injected event")
    else:
        ax.axvspan(0,end_minutes,color="orange",alpha=.13,label="Injected event")
    descriptions={"spike":"sudden single-reading error","drift":"gradually increasing offset",
                  "bias":"constant offset from original"}
    ax.set_title(f"{name.capitalize()}: {descriptions[name]} | Event {event_id}",loc="left",fontsize=11)
    ax.set_ylabel("CO2 (ppm)")
    ax.set_xlabel("Minutes from event start (0 = first injected reading)")
    ax.grid(alpha=.2)
    ax.margins(x=.04)

# Common CO2 limits support magnitude comparisons; time ranges fit each event.
all_y=np.concatenate(y_values)
lower,upper=float(np.min(all_y)),float(np.max(all_y))
padding=max(5.0,.08*(upper-lower))
axes[0].set_ylim(lower-padding,upper+padding)
# Use one shared legend rather than repeating it in every panel.
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=2,fontsize=8.5,bbox_to_anchor=(.5,.01))
fig.suptitle(f"CO2 fault examples | All predictions | Alarms: {ALARM_METHOD}",fontsize=12,y=.995)
fig.tight_layout(rect=(0,1.35/height,1,.96))
(RUN_DIR/"figures").mkdir(exist_ok=True)
panel_key="_".join(f"{p['name']}{p['event_id']}" for p in panels)
figure_path=RUN_DIR/"figures"/f"{panel_key}_all_predictions_alarm_{ALARM_METHOD}.png"
fig.savefig(figure_path,dpi=160,bbox_inches="tight")
plt.show()
plt.close(fig)

# Show selected-event injection MAE; use cells 22-23 for overall evaluation.
plot_prediction_comparison=pd.DataFrame(plot_prediction_rows)
display(plot_prediction_comparison.pivot(index="scenario",columns="name",
        values="injected_mae_vs_original_ppm").reindex(PLOT_SCENARIOS).round(2))
print('Units: ppm. This is injection-period MAE for selected events, not overall test performance or detection recall.')
print('Each event starts at minute 0; panels have separate time ranges and common CO2 limits.')
print('Colors are consistent. Ten detectors use six distinct prediction curves; EWMA/CUSUM reuse existing predictions.')
print('Constant bias means a constant observed-minus-original offset; the observed curve need not be flat.')
alarm_columns=["scenario","event_id","alarm_id","minutes_from_start","observed_ppm",
               "prediction_ppm","score_ppm","threshold_ppm","inside_injected_event"]
plot_alarm_table=pd.DataFrame(plot_alarm_rows,columns=alarm_columns)
if SHOW_ALARM_TABLE and len(plot_alarm_table):
    display(plot_alarm_table.round(2))
    print('Red crosses show alarms from:',methods[ALARM_METHOD]["label"],'; inside_injected_event=False indicates an alarm outside the selected injection.')
elif not len(plot_alarm_table):
    print('The selected method has no alarms in these local windows.')
if ALARM_METHOD.endswith("ewma"):
    print('Alarms use the EWMA statistic; score is not the current observed-minus-prediction residual.')
if ALARM_METHOD.endswith("cusum"):
    print('Alarms use the cumulative CUSUM statistic; CUSUM does not produce a separate CO2 prediction curve.')

**Code cell 25: Save run metadata and download private results**

Download an archive containing corrupted observations, evaluation references, results and plots. It contains course measurements: keep it private. The distributed code notebook contains no execution outputs or data.


In [ ]:
assert file_sha256(INPUT_CSV)==SOURCE_HASH,'The source CSV has changed; investigate.'
settings={"fault_seed":FAULT_SEED,"model_seed":MODEL_SEED,
    "spike":{"magnitude_ppm":SPIKE_PPM,"events":SPIKE_COUNT},
    "drift":{"endpoint_ppm":DRIFT_PPM,"readings":DRIFT_READINGS,"events":DRIFT_COUNT},
    "bias":{"magnitude_ppm":BIAS_PPM,"readings":BIAS_READINGS,"events":BIAS_COUNT},
    "max_gap_minutes":MAX_GAP_MINUTES,"recovery_readings":RECOVERY_READINGS,
    "threshold_quantile":THRESHOLD_QUANTILE,"ewma_alpha":EWMA_ALPHA,
    "tree_max_depth":TREE_MAX_DEPTH,"tree_min_samples_leaf":TREE_MIN_SAMPLES_LEAF,
    "cusum_k_fraction":CUSUM_K_FRACTION,"cusum_h_grid":CUSUM_H_GRID.tolist(),
    "reference_features":REFERENCE_FEATURES}
run_report={"status":"exploratory simulated-fault study; original health unknown", "audit":audit,
    "settings":settings,"split_counts":split.value_counts().to_dict(),"models":model_metadata,
    "thresholds":threshold_table.to_dict("records"),"source_unchanged":True,
    "runtime":{"python":sys.version.split()[0],"numpy":np.__version__,
               "pandas":pd.__version__,"sklearn":sklearn.__version__},
    "limitations":["Test period was previously inspected.","Only CO2 is corrupted in simulation.",
        "Fault windows use continuous records, excluding large gaps.",
        "Finite ramps return to baseline; permanent drift not tested.",
        "Recovery buffer may not capture every possible stateful-detector aftereffect.",
        "Environmental change can cause the same residual as sensor drift."]}
(RUN_DIR/"run_report.json").write_text(json.dumps(run_report,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
private_zip=RUN_DIR.parent/f"{RUN_DIR.name}_PRIVATE_results.zip"
with zipfile.ZipFile(private_zip,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path,path.relative_to(RUN_DIR))
print('Completed; the source hash is unchanged. Private backup:',private_zip.name)
if not LOCAL_CSV:
    from google.colab import files
    files.download(str(private_zip))